**Bring your own vector store: LlamaCloud Index V2 → Azure AI Search**

LlamaCloud parses; you choose where the result lives. Land LlamaCloud-parsed documents in **your
own** Azure AI Search index using the official
[`index-v2-data-sinks`](https://github.com/run-llama/index-v2-data-sinks) exporter — one call, no
indexing code, no keys — then prove the governance story and query it with LlamaIndex Retrieval
Harness primitives your agents can use.

This cookbook builds on the official LlamaIndex ↔ Azure AI Search integration, widely adopted by
enterprise customers, and extends it to LlamaCloud's Index V2 export path.

| | |
|---|---|
| **Slug** | `llamacloud-index-v2-azure-ai-search` |
| **Tags** | `azure-ai-search`, `llamaindex`, `byo-vector-store`, `retrieval`, `rag`, `entra-id`, `multi-tenant` |
| **Author** | @farzad528 |
| **Last verified** | 2026-07-17 |
| **SDK** | `index-v2-data-sinks @798cac7` (0.1.0) · `azure-search-documents==12.0.0` · `azure-identity==1.25.3` · `openai==1.109.1` |
| **REST api-version** | Azure OpenAI `2024-10-21` · Azure AI Search `2026-04-01` (SDK 12.0.0 default) |
| **Difficulty** | Intermediate |
| **Time to complete** | ~15 min |
| **Est. cost** | ~$0 incremental if you already run an Azure AI Search service (a few dozen tiny embedding calls; the demo index is torn down at the end). A dedicated Basic service is ~$75/mo if you spin one up just for this. |

## What you'll build, and why it matters

LlamaCloud's **Index V2** splits two things that vector databases usually weld together:
**parsing** (turning messy PDFs/HTML into clean, page-aware markdown) and **storage** (where the
embedded chunks live). Index V2 does the parsing and sync; the official `index-v2-data-sinks`
package lets you send the result to a store **you** own. Azure AI Search is a first-class sink —
and the one to choose when the answer has to survive a security review.

By the end you'll have run the Azure AI Search sink end-to-end against your own service — with
**no embedded API key value** — proved server-side tenant isolation on the exported index, and
layered LlamaIndex **Retrieval Harness** primitives (**List Files**, **File Grep**, **File Read**,
**Hybrid Retrieve**) on top.

**Why Azure AI Search as the sink (the customer value):**

- **Your data stays in your tenant, governed by construction.** Chunks and vectors land in *your*
  Azure AI Search — your subscription, your network, your Entra ID identities and RBAC — not a
  third-party managed index. The sink stamps every chunk with tenant and config scope, so
  isolation is a server-side filter, not client-side discipline.
- **Zero indexing code to write or maintain.** The sink auto-provisions a hybrid-ready index (HNSW
  vectors + a searchable `content` field), writes deterministic document keys, does idempotent
  upserts, per-file replace, and exposes incremental-sync snapshots. You call one function.
- **One step from agentic retrieval.** An index landed here can be wrapped as a knowledge source
  and queried by an Azure AI Search knowledge base — query decomposition, parallel subqueries,
  semantic reranking, and grounded answers. This gives Azure AI Search a documented path from
  exported chunks to Foundry IQ agentic retrieval.
- **Reuse what you already run.** Land LlamaCloud-parsed content next to everything else in Azure
  AI Search and query it with the same security, hybrid ranking, and tooling — no new datastore to
  buy, secure, or operate.
- **Agent-ready by construction.** Because the exported index keeps `parsed_directory_file_id` and
  `content`, it doubles as a filesystem-style surface agents can List / Grep / Read — not just a
  fuzzy semantic search box.

![LlamaCloud Index V2 to Azure AI Search architecture, three columns left to right. Column 1, Parse + chunk: your documents flow into LlamaCloud Index V2 (parse + sync), which emits a ParseSyncOutput into index-v2-data-sinks (chunk + embed + export) — the official exporter, one call, no indexing code, no keys. Column 2, Your Azure AI Search: the exporter writes a hybrid-ready index (HNSW vectors + BM25 content) that stamps tenant_id on every chunk so isolation is a server-side filter, runs keyless on Entra ID + RBAC in your subscription and network, and supports idempotent upserts, per-file replace, and sync snapshots. Column 3, Retrieval Harness to agents: the index exposes List Files, File Grep, File Read, and Hybrid Retrieve to your agents; the next step wraps the index as a knowledge source into a Foundry IQ knowledge base for agentic retrieval and an MCP tool.](media/llamacloud-index-v2-azure-ai-search/01-architecture.png)

## Prerequisites

- **Python 3.13+** — the `index-v2-data-sinks` package requires it.
- An **Azure AI Search** service. Any tier works for this demo; hybrid + semantic ranking on your
  own data wants Basic or higher.
- An **Azure OpenAI** (Microsoft Foundry) embedding deployment. This recipe supports
  `text-embedding-3-large` (3072-dim, default), `text-embedding-3-small` (1536-dim), or
  `text-embedding-ada-002` (1536-dim).
- **Keyless auth (the default path):** run `az login`, then assign these roles **to your own
  identity** (not to the service):
  - On the search service: `Search Service Contributor` (create/delete indexes) and
    `Search Index Data Contributor` (write and query documents)
  - On the Azure OpenAI resource: `Cognitive Services OpenAI User`

  This notebook calls the direct `/openai/v1` endpoint, so `Cognitive Services OpenAI User` is
  sufficient for embeddings. `Foundry User` is required only when you call a **Foundry project**
  endpoint for Foundry SDK, agents, or project tools.

  Role assignments take a few minutes to propagate — a `403` immediately after assigning is
  usually not a misconfiguration. Wait and retry.
- **~15 minutes.** The demo index is deleted at the end, so incremental cost is negligible.

Set these in your shell or a local `.env` next to this notebook (the repo `.gitignore` excludes `.env`):

```bash
SEARCH_ENDPOINT=https://<your-search-service>.search.windows.net
AOAI_ENDPOINT=https://<your-foundry-resource>.openai.azure.com
AOAI_EMBEDDING_DEPLOYMENT=<your-deployment-for-large-small-or-ada-002>
SEARCH_API_KEY=      # optional fallback ONLY if you can't get RBAC — leave unset for keyless
AOAI_API_KEY=        # optional fallback ONLY if you can't get RBAC — leave unset for keyless
```

> No API key value is embedded in this notebook. Key environment variables remain an optional
> fallback; leave them unset and the sink uses `DefaultAzureCredential`, so the whole write path
> runs on your Entra ID identity.

## Setup

Pinned to an exact commit — `index-v2-data-sinks` ships from GitHub (not PyPI) and has no releases,
so an unpinned install would drift the moment the repo moves. The Azure SDKs are pinned explicitly
rather than left to the sink's `>=` floors.

In [ ]:
%%capture
# index-v2-data-sinks is published on GitHub, not PyPI, and needs Python 3.13+.
try:
    import index_v2_data_sinks  # noqa: F401
except ImportError:
    %pip install --quiet "index-v2-data-sinks@git+https://github.com/run-llama/index-v2-data-sinks@798cac7e25f7485aa78b5f666957af41cae62c2f" "azure-search-documents==12.0.0" "azure-identity==1.25.3" "openai==1.109.1" "beautifulsoup4==4.13.3" "python-dotenv==1.2.2"

## Configuration — the only cell you edit

Every knob lives here, named, so the rest of the notebook has no magic numbers. This is the cell the
copy-paste reader changes and nothing else.

In [ ]:
# --- Configuration ---------------------------------------------------------
# Index naming. The sink provisions ONE index per embedding model; the default
# model keeps the base name, others get a sanitized suffix.
INDEX_PREFIX = "forgebook-index-v2-sink"

# Which model the sink sizes the index for. The Azure OpenAI deployment configured
# below must be the same base model, or query-time vectors and the index dimension will diverge.
EMBEDDING_MODELS = {
    "openai:text-embedding-3-large": 3072,
    "openai:text-embedding-3-small": 1536,
    "openai:text-embedding-ada-002": 1536,
}
EMBEDDING_MODEL = "openai:text-embedding-3-large"  # Change this with your deployment.
EMBEDDING_DIM = EMBEDDING_MODELS[EMBEDDING_MODEL]
AOAI_API_VERSION = "2024-10-21"   # Azure OpenAI data-plane api-version
EMBED_BATCH = 16                  # chunks embedded per Azure OpenAI call


def sink_index_name(prefix, embedding_model):
    """Match the official sink's one-index-per-model naming convention."""
    if embedding_model == "openai:text-embedding-3-small":
        return prefix
    suffix = embedding_model.replace(":", "-").replace("/", "-").replace(".", "-").replace("_", "-").lower()
    return f"{prefix}-{suffix}"


INDEX_NAME = sink_index_name(INDEX_PREFIX, EMBEDDING_MODEL)

# How the sink chunks each document: "fast" | "sentence" | "token".
CHUNK_STRATEGY = "sentence"

# Tenant + export-config labels written onto every chunk. In production these are
# your LlamaCloud project id and export-config id; here they're the scope the sink
# filters on for tenant isolation, per-file replace, and snapshots.
PROJECT_ID = "forgebook-demo"     # -> tenant_id on every document
CONFIG_ID = "forgebook-config"    # -> export_config_id; the snapshot/delete scope

# Retrieval Harness knobs.
TOP_K = 5        # final hits returned by Hybrid Retrieve
K_NEAREST = 50   # vector candidates fused before ranking; raise for recall
GREP_TOP = 50    # max chunks File Grep scans in one file
READ_TOP = 1000  # whole-file ceiling for File Read
LIST_TOP = 1000  # max docs List Files scans to build the map

# The demo corpus: three Paul Graham essays, each becomes one "file".
ESSAYS = {
    "great-work": "https://paulgraham.com/greatwork.html",
    "superlinear": "https://paulgraham.com/superlinear.html",
    "how-to-do-what-you-love": "https://paulgraham.com/love.html",
}
# ---------------------------------------------------------------------------
print(f"Config set. Index '{INDEX_NAME}', {len(ESSAYS)} source files, top_k={TOP_K}.")

## Connect to your services — keyless by default

We resolve one `DefaultAzureCredential` for everything: your own Azure AI Search clients, the
Azure OpenAI embedding client, and the sink (its `AzureSearchService` reads
`DEFAULT_INDEX_AZURE_SEARCH_*` env vars and falls back to Entra ID when no key is set). If you set
the fallback keys in `.env`, they win — but the default path never touches one.

In [ ]:
import os

from azure.core.credentials import AzureKeyCredential
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from dotenv import load_dotenv
from openai import AzureOpenAI

load_dotenv(override=True)


def env(name, *, required=True, default=None):
    value = os.getenv(name, default)
    if required and not value:
        raise RuntimeError(f"Missing required env var: {name}. See Prerequisites.")
    return value or None


SEARCH_ENDPOINT = env("SEARCH_ENDPOINT")
AOAI_ENDPOINT = env("AOAI_ENDPOINT").split("/openai/", 1)[0].rstrip("/")
EMBED_DEPLOYMENT = env(
    "AOAI_EMBEDDING_DEPLOYMENT",
    required=False,
    default="text-embedding-3-large",
)  # Use the same base model as EMBEDDING_MODEL.

# Optional key fallback — leave both unset for the keyless default path.
SEARCH_API_KEY = env("SEARCH_API_KEY", required=False)
AOAI_API_KEY = env("AOAI_API_KEY", required=False)

# The sink's AzureSearchService reads its target from these env vars. With no
# API key set, it resolves DefaultAzureCredential — the same identity as ours.
os.environ["DEFAULT_INDEX_AZURE_SEARCH_ENDPOINT"] = SEARCH_ENDPOINT
os.environ["DEFAULT_INDEX_AZURE_SEARCH_INDEX_PREFIX"] = INDEX_PREFIX
if SEARCH_API_KEY:
    os.environ["DEFAULT_INDEX_AZURE_SEARCH_API_KEY"] = SEARCH_API_KEY

# Credential for the clients this notebook creates directly.
search_credential = AzureKeyCredential(SEARCH_API_KEY) if SEARCH_API_KEY else DefaultAzureCredential()

if AOAI_API_KEY:
    aoai = AzureOpenAI(
        azure_endpoint=AOAI_ENDPOINT,
        api_key=AOAI_API_KEY,
        api_version=AOAI_API_VERSION,
    )
else:
    aoai = AzureOpenAI(
        azure_endpoint=AOAI_ENDPOINT,
        azure_ad_token_provider=get_bearer_token_provider(
            DefaultAzureCredential(),
            "https://cognitiveservices.azure.com/.default",
        ),
        api_version=AOAI_API_VERSION,
    )


def embed(texts):
    """Embed with your Azure OpenAI deployment, batched."""
    vectors = []
    for i in range(0, len(texts), EMBED_BATCH):
        response = aoai.embeddings.create(
            model=EMBED_DEPLOYMENT,
            input=texts[i : i + EMBED_BATCH],
        )
        vectors.extend(item.embedding for item in response.data)
    return vectors


auth_mode = "API key (fallback)" if SEARCH_API_KEY else "Entra ID (keyless)"
print(f"Auth: {auth_mode} | embeddings: {EMBED_DEPLOYMENT} | sink will provision: {INDEX_NAME}")

## Verify setup

Fail loud *now* if the credential or a deployment is wrong, instead of 20 cells later inside an
export call. A `403` here on the keyless path almost always means a role hasn't finished
propagating — see Prerequisites.

In [ ]:
from azure.search.documents.indexes import SearchIndexClient

probe = embed(["setup probe"])[0]
assert len(probe) == EMBEDDING_DIM, (
    f"Embedding returned {len(probe)} dims, expected {EMBEDDING_DIM} — is AOAI_EMBEDDING_DEPLOYMENT "
    f"a text-embedding-3-large deployment?"
)

index_client = SearchIndexClient(SEARCH_ENDPOINT, search_credential)
existing = list(index_client.list_index_names())
print(f"Setup OK — embeddings return {len(probe)} dims; Azure AI Search reachable ({len(existing)} indexes).")

## How you'd run this in production — one call

Point the sink at a LlamaCloud Index V2 directory and it pulls the parse outputs, chunks, embeds, and
writes to Azure AI Search for you:

```python
from index_v2_data_sinks.pipeline import export_pipeline

await export_pipeline(
    export_to="azure_search",
    directory_name="my-index-directory",
    project_id="<llamacloud-project-id>",
    config_id="<export-config-id>",
    embedding_model="openai:text-embedding-3-small",
    chunking_strategy="sentence",
)
```

That needs a `LLAMA_CLOUD_API_KEY`, a parsed Index V2 directory, and an embedding-provider key. To keep
this notebook runnable on *just* Azure — no LlamaCloud account — we call the sink's own components
directly. Internally `export_pipeline` is exactly:

> build a `ParseSyncOutput` → chunk it (`ChunkerService`) → build `ExportedOutputV1` records → `AzureSearchExporter.export(...)`

We do the same, sourcing the `ParseSyncOutput` locally and the embeddings from Azure OpenAI.
**Everything that touches Azure AI Search is the unmodified official package.**

## Step 1 — Build parse outputs and chunk them (with the package)

We download three Paul Graham essays and wrap each as a `ParseSyncOutput` — the shape LlamaCloud hands
the sink. Then we chunk with the package's `ChunkerService` and build `ExportedOutputV1` records, the
exact calls `export_pipeline` makes internally.

In [ ]:
import re
import urllib.request

from bs4 import BeautifulSoup
from index_v2_data_sinks.chunking import ChunkerService
from index_v2_data_sinks.schema import ExportedOutputV1, ParsedPage, ParseSyncOutputV1


def fetch_text(url):
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request, timeout=30) as response:
        html = response.read().decode("utf-8", errors="ignore")
    return re.sub(r"\n{3,}", "\n\n", BeautifulSoup(html, "html.parser").get_text("\n")).strip()


chunker = ChunkerService()
files = {}  # parsed_directory_file_id -> list[ExportedOutputV1]
for file_id, url in ESSAYS.items():
    page_text = fetch_text(url)
    parse_output = ParseSyncOutputV1(pages=[ParsedPage(page_number=1, text=page_text)])
    # Single-page docs, so the chunker input is the page text itself.
    # ExportedOutputV1.build maps each chunk back to its page range internally.
    chunks = chunker.chunk(page_text, CHUNK_STRATEGY)
    files[file_id] = [
        ExportedOutputV1.build(parse_output, PROJECT_ID, CONFIG_ID, chunk, file_id, metadata={"file_name": file_id})
        for chunk in chunks
    ]
    print(f"{file_id:>26}: {len(files[file_id]):3d} chunks")

Each essay is now a list of `ExportedOutputV1` records carrying the chunk text, page range, and the
tenant/config scope — everything the sink needs, and nothing about Azure yet.

## Step 2 — Export to Azure AI Search via the official sink

`AzureSearchExporter.export(...)` provisions the index on first call (HNSW vectors, a searchable
`content` field, filterable scoping fields), then upserts the chunks with their embeddings. We embed
each file with Azure OpenAI and hand the vectors to the exporter. `export()` is async, so we `await`
it. First we delete any leftover index so this run's counts are exact.

In [ ]:
from index_v2_data_sinks.export.azure_search import AzureSearchExporter

# Start from a clean index so the counts below are exact on a re-run.
if INDEX_NAME in index_client.list_index_names():
    index_client.delete_index(INDEX_NAME)

exporter = AzureSearchExporter()

total = 0
for file_id, outputs in files.items():
    embeddings = embed([o.content for o in outputs])
    await exporter.export(
        outputs,
        project_id=PROJECT_ID,
        embeddings=embeddings,
        embedding_model=EMBEDDING_MODEL,
    )
    total += len(outputs)
    print(f"exported {file_id}: {len(outputs)} chunks")

print(f"\nExported {total} chunks via the official AzureSearchExporter into '{INDEX_NAME}'.")

That's the whole write path — no index schema, no field definitions, no upload loop, and no key
written by you. The sink owned all of it, on your Entra ID identity.

## Step 3 — Inspect the auto-provisioned index

Let's see what the sink actually built. The schema and the deterministic
`{parsed_directory_file_id}__{chunk_index}` document key both come straight from the package. Note
the scoping fields — `tenant_id` and `export_config_id` — on every document; we'll use them next.

In [ ]:
import time

from azure.search.documents import SearchClient

search_client = SearchClient(SEARCH_ENDPOINT, INDEX_NAME, search_credential)


def wait_for_docs(client, target, timeout=30):
    """Azure AI Search indexing is near-real-time, not instant — poll until
    the document count reaches the target (or the timeout passes)."""
    for _ in range(timeout):
        if client.get_document_count() >= target:
            break
        time.sleep(1)
    return client.get_document_count()


count = wait_for_docs(search_client, total)
schema = index_client.get_index(INDEX_NAME)
print("Index fields:", [f.name for f in schema.fields])
print("Documents:", count)

doc = next(iter(search_client.search(
    search_text="*",
    select=["id", "parsed_directory_file_id", "tenant_id", "chunk_index", "content"],
)))
print("\nRaw document:")
print("  id:", doc["id"])
print("  tenant_id:", doc["tenant_id"], "| file:", doc["parsed_directory_file_id"], "| chunk_index:", doc["chunk_index"])
print("  content:", " ".join(doc["content"].split())[:160], "...")

Note the `content` field (searchable, for BM25) and `embedding` (the vector) sitting side by side —
that's what makes the index hybrid-ready out of the box — plus `tenant_id` stamped on every chunk.

## Step 4 — Prove the governance story: server-side tenant isolation

"Governed" should be a demo, not an adjective. The sink stamped every document with `tenant_id`
(your LlamaCloud project) and `export_config_id`. Scope every query with a filter on those fields
and isolation is enforced **in the service** — a tenant can never see another tenant's chunks, no
matter what the query text says.

In [ ]:
same_query = "What does it take to do great work?"

mine = list(search_client.search(
    search_text=same_query,
    filter=f"tenant_id eq '{PROJECT_ID}'",
    select=["parsed_directory_file_id", "chunk_index"],
    top=TOP_K,
))
someone_else = list(search_client.search(
    search_text=same_query,
    filter="tenant_id eq 'another-tenant'",
    select=["parsed_directory_file_id", "chunk_index"],
    top=TOP_K,
))
print(f"tenant '{PROJECT_ID}': {len(mine)} hits  |  tenant 'another-tenant': {len(someone_else)} hits")

Same query, zero leakage — and the filter runs server-side, not in your client code. Combined with
the keyless Entra ID identity this notebook already runs under, those are the first two things a
security review asks about. The next maturity step is per-user
[security trimming](https://learn.microsoft.com/en-us/azure/search/search-security-trimming-for-azure-search)
(ACL filtering at query time), which changes the schema and query path — that's its own cookbook.

## Step 5 — LlamaIndex Retrieval Harness primitives on the sink index

LlamaIndex's **Retrieval Harness** defines the primitives an agent needs over a corpus: **List
Files**, **File Grep**, **File Read**, and a high-recall **Hybrid Retrieve**. Because the sink
lands `parsed_directory_file_id`, `chunk_index`, and a searchable `content` field next to the
vector, all four run natively against Azure AI Search — no extra schema, no second store. We wrap
the raw SDK so every result stays inspectable (scores included).

> The sink doesn't provision a semantic-reranker config, so **Hybrid Retrieve** here is vector + BM25
> fusion. Add a semantic configuration to the index if you want Azure's reranker on top.

In [ ]:
from azure.search.documents.models import VectorizedQuery


def _escape_odata(value):
    """OData string literals escape embedded single quotes by doubling them."""
    return value.replace("'", "''")


class SinkRetrievalHarness:
    """Retrieval Harness primitives over an index-v2-data-sinks Azure AI Search index."""

    def __init__(self, client, embed_fn):
        self.client = client
        self.embed_fn = embed_fn
        self.select = ["parsed_directory_file_id", "chunk_index", "content"]

    def hybrid_retrieve(self, query, top_k=TOP_K, k_nearest=K_NEAREST):
        """High-recall first pass: vector + BM25 fusion over the sink index."""
        vector = self.embed_fn([query])[0]
        results = self.client.search(
            search_text=query,
            vector_queries=[VectorizedQuery(vector=vector, k_nearest_neighbors=k_nearest, fields="embedding")],
            select=self.select,
            top=top_k,
        )
        return [self._row(r) for r in results]

    def list_files(self, top=LIST_TOP):
        """Map the corpus: every parsed_directory_file_id and its chunk count."""
        results = self.client.search(search_text="*", select=["parsed_directory_file_id"], top=top)
        counts = {}
        for r in results:
            counts[r["parsed_directory_file_id"]] = counts.get(r["parsed_directory_file_id"], 0) + 1
        return [{"file": name, "chunks": n} for name, n in sorted(counts.items())]

    def file_grep(self, file_id, term, top=GREP_TOP):
        """Server-side term scan of one file. Matching goes through the query
        analyzer — tokenized and case-insensitive, not byte-exact."""
        odata = (
            f"parsed_directory_file_id eq '{_escape_odata(file_id)}' and "
            f"search.ismatch('{_escape_odata(term)}', 'content', 'full', 'any')"
        )
        results = self.client.search(search_text="*", filter=odata, select=self.select, top=top)
        return sorted((self._row(r) for r in results), key=lambda d: d["chunk_index"])

    def file_read(self, file_id, top=READ_TOP):
        """Pull a whole file back in chunk order."""
        results = self.client.search(
            search_text="*",
            filter=f"parsed_directory_file_id eq '{_escape_odata(file_id)}'",
            select=self.select,
            top=top,
        )
        return sorted((self._row(r) for r in results), key=lambda d: d["chunk_index"])

    def _row(self, r):
        return {
            "file": r.get("parsed_directory_file_id"),
            "chunk_index": r.get("chunk_index"),
            "score": r.get("@search.score"),
            "text": r.get("content"),
        }


harness = SinkRetrievalHarness(search_client, embed)


def show(rows, n=200):
    for r in rows:
        head = f"[{r['file']} #{r['chunk_index']:>3}]"
        if r.get("score") is not None:
            head += f"  score={r['score']:.4f}"
        print(head)
        print("  ", " ".join((r.get("text") or "").split())[:n], "...\n")


print("Harness ready:", ["hybrid_retrieve", "list_files", "file_grep", "file_read"])

### Hybrid Retrieve — find where an answer lives

One call embeds the query, runs a vector search, and fuses it with BM25. This is the agent's fast,
high-recall first pass over the whole corpus.

In [ ]:
show(harness.hybrid_retrieve("What does it take to do great work?"))

Top hits are the essay's thesis chunks, with fused relevance scores you can inspect directly.

### List Files — map the corpus

Before an agent reads or greps, it needs the directory. `list_files` returns every file and its chunk
count, straight from the `parsed_directory_file_id` field the sink wrote.

In [ ]:
for f in harness.list_files():
    print(f"{f['file']:>26}  ({f['chunks']} chunks)")

Three files, each with its chunk count — the corpus as a filesystem listing.

### File Grep — term scan inside one file

When the agent needs occurrences of a specific term, it scans a single file server-side
(`search.ismatch` scoped by an OData filter), spending no tokens on irrelevant chunks. One honesty
note: `search.ismatch` runs through the **query analyzer**, so matching is tokenized and
case-insensitive — closer to `grep -iw` than byte-exact `grep`. For byte-exact matching you'd add
a field with the `keyword` analyzer.

In [ ]:
hits = harness.file_grep("superlinear", "exponential")
print(f"'exponential' -> {len(hits)} chunk(s) in superlinear\n")
show(hits[:3])

Term hits only, confined to the file you named — the precision an agent needs to cite a specific
chunk rather than a fuzzy neighborhood.

### File Read — recover whole-file context

Top-k retrieval fragments documents. When a chunk cuts off mid-thought, the agent reads the file back
**in order** to recover surrounding context.

In [ ]:
chapter = harness.file_read("how-to-do-what-you-love")
print(f"Reassembled {len(chapter)} chunks in order:\n")
for r in chapter[:3]:
    print(f"#{r['chunk_index']:>3}  {' '.join((r['text'] or '').split())[:200]}...")

The file comes back as an ordered sequence — chunk 0, 1, 2, … — because the sink stored a
`chunk_index` on every document.

## Step 6 — Idempotent re-export and incremental sync

Re-running the export for a file is safe: the deterministic key + `merge_or_upload` keeps the chunk
count stable instead of duplicating. The sink also exposes `list_snapshots()` — one record per file —
so an orchestrator can diff against the directory and re-export only what changed.

In [ ]:
before = search_client.get_document_count()

again = files["superlinear"]
await exporter.export(
    again, project_id=PROJECT_ID, embeddings=embed([o.content for o in again]), embedding_model=EMBEDDING_MODEL
)

# merge_or_upload with deterministic keys means the count must NOT grow. Poll for
# a few seconds so a would-be duplicate has time to surface before we compare.
deadline = time.time() + 8
while time.time() < deadline and search_client.get_document_count() <= before:
    time.sleep(1)
print(f"doc count before re-export: {before}  |  after: {search_client.get_document_count()}  (no duplicates)")

snapshots = await exporter.list_snapshots(export_config_id=CONFIG_ID, embedding_model=EMBEDDING_MODEL)
print("snapshots (one per file):", sorted(s.parsed_directory_file_id for s in snapshots))

Same document count after re-exporting a file, and a snapshot per file — the primitives an incremental
sync loop is built from. (Note `list_snapshots` needs `embedding_model` to resolve the per-model index
name; omit it and it queries the wrong index.)

## Make it yours

Everything above ran against three sample essays and demo labels. Here's what changes for your data:

| Config | Sample value | What you set | Watch out for |
|---|---|---|---|
| `ESSAYS` | 3 Paul Graham essays | Your own docs — or, in production, drop this entirely and use `export_pipeline` against a LlamaCloud directory | Each key becomes a `parsed_directory_file_id`; keep them stable across runs so re-export replaces instead of duplicating |
| `EMBEDDING_MODEL` | `openai:text-embedding-3-large` | The model id the sink sizes the index for | **Must match your actual embedding model** — a mismatch silently returns garbage. The Azure deployment and this id must be the same model |
| `AOAI_EMBEDDING_DEPLOYMENT` | `text-embedding-3-large` | Your Azure OpenAI deployment name | Use the same base model as `EMBEDDING_MODEL`: large = 3072 dims; small or ada-002 = 1536 |
| `INDEX_PREFIX` | `forgebook-index-v2-sink` | Your index base name | Final index = `{prefix}-{sanitized-model}`; one index per embedding model |
| `PROJECT_ID` / `CONFIG_ID` | demo labels | Your LlamaCloud project + export-config ids | `PROJECT_ID` is the `tenant_id` isolation scope; `CONFIG_ID` scopes per-file delete and `list_snapshots` — keep both stable |
| `TOP_K` / `K_NEAREST` | `5` / `50` | Tune on your queries | Raise `K_NEAREST` for recall on hard queries; each point costs a little latency |
| Auth | Keyless (`az login` + RBAC) | Fallback: set `SEARCH_API_KEY` / `AOAI_API_KEY` in `.env` | Role assignments take a few minutes to propagate |

**What changes at scale:** this ran over 3 files and 60 chunks. At thousands of files, drive exports
from `export_pipeline` against a real LlamaCloud directory and use `list_snapshots()` to sync only
changed files. Add a semantic configuration to the index if you want Azure AI Search's Semantic
Ranker on top of the vector + BM25 fusion shown here — it's also the prerequisite for wrapping this
index as a knowledge source (see Next steps).

## Troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `ModuleNotFoundError: index_v2_data_sinks` | Python < 3.13, or install skipped | The package requires Python 3.13+. Re-run the setup cell on a 3.13 kernel |
| `403 Forbidden` on the keyless path | RBAC role not assigned to *you*, or still propagating | Assign `Search Service Contributor` + `Search Index Data Contributor` to your own identity; wait ~5 min and retry |
| Embedding probe asserts wrong dim | Deployment and `EMBEDDING_MODEL` differ | Use a matching pair: large = 3072 dims; small or ada-002 = 1536, then rebuild the index |
| `list_snapshots()` returns `[]` | Called without `embedding_model` → resolves the base index name, not the per-model one | Always pass `embedding_model=EMBEDDING_MODEL` |
| Queries return 0 results right after export | Azure AI Search indexing is near-real-time, not instant | The inspect cell polls with `wait_for_docs`; wait a moment and re-run the query |
| File Grep matches more (or less) than expected | `search.ismatch` runs through the query analyzer — tokenized, case-insensitive | Expected behavior; for byte-exact matching add a `keyword`-analyzer field |
| Retrieval feels random | Query-time and index-time embedding models differ | Make `EMBED_DEPLOYMENT` and `EMBEDDING_MODEL` the same model |

## Teardown

The index lives in *your* service. Delete it when you're done, or set `DELETE_INDEX = False` to keep
it and browse the raw documents in the Azure portal.

In [ ]:
DELETE_INDEX = True
if DELETE_INDEX:
    index_client.delete_index(INDEX_NAME)
    print(f"Deleted index '{INDEX_NAME}'.")
else:
    print(f"Kept index '{INDEX_NAME}' — inspect it in Azure portal > Search service > Indexes.")

The Azure AI Search *service* is not deleted — it bills whether or not it's queried. If you spun one
up just for this, remove it from the portal or with `az search service delete`.

## Next steps

- **Run the real pipeline** — swap the local parse output for `export_pipeline(export_to="azure_search", ...)`
  pointed at a LlamaCloud Index V2 directory. The Azure AI Search half is identical to what you just ran.
- **Add Azure AI Search's Semantic Ranker** — attach a semantic configuration to the index and
  switch Hybrid Retrieve to `query_type="semantic"` for 0–4 reranker scores and extractive captions.
- **Graduate to agentic retrieval** — wrap the exported index as a
  [search index knowledge source](https://learn.microsoft.com/en-us/azure/search/agentic-knowledge-source-how-to-search-index)
  (it needs the semantic configuration from the previous step) and add it to a
  [knowledge base](https://learn.microsoft.com/en-us/azure/search/agentic-retrieval-how-to-create-knowledge-base):
  complex questions get decomposed into parallel subqueries, semantically reranked, and answered
  with citations — over the same index the sink just built. Every knowledge base is also a
  standalone MCP server, so agents in Foundry Agent Service, GitHub Copilot, or Claude can call it
  as a tool directly.